In the following set of experiments, we aim to generate meaningful benchmarks for the dose escalation methods we would like to study.

### Utilities

In [1]:
import numpy as np
from doseescalation.dose_escalator import (
    CRMDoseEscalator, 
    DoseEscalatorBase,
    ThreePlusThreeDoseEscalator, 
    UCBDoseEscalator,
)
from doseescalation.estimator import (
    AveragingEstimator,
)
from doseescalation.evaluate import (
    plot_dose_proposals, 
    plot_acc_progression,
    plot_n_dles,
    simulate,
)
from doseescalation.simulated_env import SimulatedEnv
from typing import Callable, Sequence

In [2]:
def a_key(a):
    return f"a = {a:.1f}"

def cohort_key(cohort):
    return f"Cohort {cohort + 1}"

In [3]:
def dose_toxic_curve(dose, a):
    return np.power((np.tanh(dose) + 1) / 2, a)

def inv_dose_toxic(p_dle, a):
    return np.arctanh(2 * np.power(p_dle, 1 / a) - 1)

In [4]:
N_LEVELS = 6
P_DLE_LEVELS = {
    0.4: [0.3, 0.4, 0.53, 0.62, 0.76, 0.87],
    1.0: [0.05, 0.1, 0.2, 0.3, 0.5, 0.7],
    1.3: [0.02, 0.05, 0.12, 0.3, 0.41, 0.63],
    3.4: [0.01, 0.02, 0.04, 0.08, 0.16, 0.3],
}
DOSE_LEVELS = {
    a: [inv_dose_toxic(v, a) for v in vs] 
    for a, vs in P_DLE_LEVELS.items()
}
N_A = len(DOSE_LEVELS.keys())
TTL = 0.3
N_TRIALS = 300

# To add more algorithms, extend the following list to include them.
ALGOS = ["3 + 3", "CRM", "UCB"]

COHORT_SIZE = 3
CORRECT_MTDS = {
    a_key(0.4): 0, 
    a_key(1.0): 3, 
    a_key(1.3): 3, 
    a_key(3.4): 5,
}

In [5]:
def run_simulations(
    dose_escalator: DoseEscalatorBase,
    dose_levels: Sequence[float],
    dose_toxic_curve: Callable,
    cohort_size: int, 
    n_cohorts: int,
):
    env = SimulatedEnv(dose_levels, dose_toxic_curve)
    return simulate(
        cohort_sizes=[cohort_size] * n_cohorts, 
        dose_escalator=dose_escalator, 
        env=env
    )

### Simulations

We run 2 main sets of simulations, differing in the number of cohorts used in their experiments. 

The more realistic setting uses around 10^1 cohorts while the setting for asymptotic behaviours uses more than 10^2.

#### Realistic

In [6]:
N_REAL_COHORTS = 9

Run the simulations and collect results in a number of dictionaries, for later analysis.

In [7]:
a_algo_dli_map = {
    a_key(a): {
        algo: [] for algo in ALGOS
    } for a in DOSE_LEVELS.keys()
}
cohort_algo_dli_map = {
    a_key(a): {
        cohort_key(cohort): {
            algo: [] for algo in ALGOS
        } for cohort in range(N_REAL_COHORTS)
    } for a in DOSE_LEVELS.keys()
}
a_algo_n_dle_map = {
    a_key(a): {
        algo: [] for algo in ALGOS
    } for a in DOSE_LEVELS.keys()
}

def add_simulations(dose_escalator, a, algo):
    dlis, n_dles = run_simulations(
        dose_escalator, 
        dose_levels,
        lambda dose: dose_toxic_curve(dose, a), 
        COHORT_SIZE,
        n_cohorts=N_REAL_COHORTS,
    )
    a_algo_dli_map[a_key(a)][algo].append(dlis[-1])
    a_algo_n_dle_map[a_key(a)][algo].append(sum(n_dles))
    for cohort, dli in enumerate(dlis):
        cohort_algo_dli_map[a_key(a)][cohort_key(cohort)][algo].append(dli)
        
for a, dose_levels in DOSE_LEVELS.items():
    for _ in range(N_TRIALS):
        # To include more dose escalators, add a similar simulation call here.
        tpt_dose_escalator = ThreePlusThreeDoseEscalator(
            dose_levels=dose_levels
        )
        add_simulations(tpt_dose_escalator, a, ALGOS[0])
        
        crm_dose_escalator = CRMDoseEscalator(
            dose_levels=dose_levels, 
            target_toxicity_level=TTL, 
            estimator=AveragingEstimator(), 
            conservative=False
        )
        add_simulations(crm_dose_escalator, a, ALGOS[1])
        
        ucb_dose_escalator = UCBDoseEscalator(
            dose_levels=dose_levels,
            target_toxicity_level=TTL,
            estimator=AveragingEstimator(),
            ucb_coefficient=0.1,
        )
        add_simulations(ucb_dose_escalator, a, ALGOS[2])

Plot the final proposals made (for the last cohort) by each of the algorithm, across the trial runs.

In [8]:
plot_dose_proposals(
    N_LEVELS, 
    N_TRIALS, 
    a_algo_dli_map, 
    CORRECT_MTDS,
    img_path=f"plots/real_algo_dose_proposals.png",
)

Plot the proposals made for each cohort, so that we can see the time evolution of our `DoseEscalator`'s proposals. This is also across all the trial runs, and one plot for each dose-toxicity curve.

In [9]:
for a in DOSE_LEVELS.keys():
    correct_mtds = {
        cohort_key(cohort): CORRECT_MTDS[a_key(a)]
        for cohort in range(N_REAL_COHORTS)
    }
    plot_dose_proposals(
        N_LEVELS, 
        N_TRIALS, 
        cohort_algo_dli_map[a_key(a)], 
        correct_mtds, 
        unit_width=100,
        title_text="Number of dose allocations at each cohort",
        show_fig=False,
        img_path=f"plots/real_{a_key(a)}_dose_proposal_progression.png"
    )

Plot the distribution of dose limiting events across all the trial runs and cohorts.

In [10]:
plot_n_dles(
    a_algo_n_dle_map, 
    img_path=f"plots/real_algo_n_dles.png",
)

#### Asymptotic

In [11]:
N_ASYM_COHORTS = 300

Run the simulations and collect results in a number of dictionaries, for later analysis.

In [12]:
a_algo_dli_map = {
    a_key(a): {
        algo: [] for algo in ALGOS
    } for a in DOSE_LEVELS.keys()
}
cohort_algo_dli_map = {
    a_key(a): {
        cohort_key(cohort): {
            algo: [] for algo in ALGOS
        } for cohort in range(N_ASYM_COHORTS)
    } for a in DOSE_LEVELS.keys()
}

def add_simulations(dose_escalator, a, algo):
    dlis, _ = run_simulations(
        dose_escalator, 
        dose_levels,
        lambda dose: dose_toxic_curve(dose, a), 
        COHORT_SIZE,
        n_cohorts=N_ASYM_COHORTS,
    )
    a_algo_dli_map[a_key(a)][algo].append(dlis[-1])
    for cohort, dli in enumerate(dlis):
        cohort_algo_dli_map[a_key(a)][cohort_key(cohort)][algo].append(dli)
        
for a, dose_levels in DOSE_LEVELS.items():
    for _ in range(N_TRIALS):
        # To include more dose escalators, add a similar simulation call here.
        tpt_dose_escalator = ThreePlusThreeDoseEscalator(
            dose_levels=dose_levels
        )
        add_simulations(tpt_dose_escalator, a, ALGOS[0])
        
        crm_dose_escalator = CRMDoseEscalator(
            dose_levels=dose_levels, 
            target_toxicity_level=TTL, 
            estimator=AveragingEstimator(), 
            conservative=False
        )
        add_simulations(crm_dose_escalator, a, ALGOS[1])
        
        ucb_dose_escalator = UCBDoseEscalator(
            dose_levels=dose_levels,
            target_toxicity_level=TTL,
            estimator=AveragingEstimator(),
            ucb_coefficient=0.1,
        )
        add_simulations(ucb_dose_escalator, a, ALGOS[2])

Plot the final proposals made (for the last cohort) by each of the algorithm, across the trial runs.

In [13]:
plot_dose_proposals(
    N_LEVELS, 
    N_TRIALS, 
    a_algo_dli_map, 
    CORRECT_MTDS,
    img_path=f"plots/asym_algo_dose_proposals.png",
)

Plot the proposal accuracy (whether it matches the MTD) for each cohort so that we can see the time evolution of our `DoseEscalator`'s proposals. 

This is also across all the trial runs, and one plot for each dose-toxicity curve.

In [14]:
plot_acc_progression(
    N_ASYM_COHORTS, 
    cohort_algo_dli_map,
    CORRECT_MTDS,
    show_fig=False, 
    img_path=f"plots/asym_dose_proposal_acc_progression.png",
)